# Financial News Analyzer - Getting Started Notebook

This notebook demonstrates the core functionality of the Financial News Analyzer.

## Setup

In [ ]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ Setup complete!")

## 1. Initialize Agents

Let's initialize our multi-agent system.

In [ ]:
from langchain.llms import OpenAI
from src.agents.research_agent import ResearchAgent
from src.agents.sentiment_agent import SentimentAgent
from src.agents.risk_agent import RiskAgent
from src.tools.news_tool import create_news_tools

# Initialize LLM
llm = OpenAI(temperature=0.3)

# Create tools
news_tools = create_news_tools()

# Initialize agents
research_agent = ResearchAgent(llm=llm, tools=news_tools, verbose=True)
sentiment_agent = SentimentAgent(llm=llm, verbose=True)
risk_agent = RiskAgent(llm=llm, verbose=True)

print("✅ Agents initialized!")

## 2. Research Agent Demo

Let's use the Research Agent to gather information about a stock.

In [ ]:
# Define stock to analyze
symbol = "AAPL"
days_back = 7

# Run research
research_results = research_agent.execute({
    "symbol": symbol,
    "days_back": days_back
})

print(f"\n📊 Research Results for {symbol}")
print("=" * 60)
print(research_results['findings'])

## 3. Sentiment Analysis

Now let's analyze the sentiment of the gathered news.

In [ ]:
# Prepare sample articles for sentiment analysis
sample_articles = [
    "Apple reports strong Q4 earnings, beating analyst expectations.",
    "New iPhone launch receives positive reviews from critics.",
    "Regulatory concerns in EU markets may impact Apple's operations."
]

# Run sentiment analysis
sentiment_results = sentiment_agent.execute({
    "symbol": symbol,
    "articles": sample_articles
})

print(f"\n💭 Sentiment Analysis for {symbol}")
print("=" * 60)
print(f"Overall Sentiment: {sentiment_results['overall_sentiment']}")
print(f"Sentiment Score: {sentiment_results['sentiment_score']:.2f}")
print(f"Confidence: {sentiment_results['confidence']:.2f}")

### Visualize Sentiment

In [ ]:
# Create sentiment gauge
fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=sentiment_results['sentiment_score'] * 100,
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': "Sentiment Score"},
    delta={'reference': 50},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "darkblue"},
        'steps': [
            {'range': [0, 40], 'color': "lightcoral"},
            {'range': [40, 60], 'color': "lightyellow"},
            {'range': [60, 100], 'color': "lightgreen"}
        ],
    }
))

fig.show()

## 4. Risk Assessment

Let's assess the risks for this stock.

In [ ]:
# Run risk assessment
risk_results = risk_agent.execute({
    "symbol": symbol,
    "news_data": research_results,
    "market_data": {"volatility": "moderate"},
    "sentiment": sentiment_results
})

print(f"\n⚠️ Risk Assessment for {symbol}")
print("=" * 60)
print(f"Overall Risk Score: {risk_results['overall_risk_score']:.2f}")
print(f"Risk Level: {risk_results['risk_level']}")
print(f"\nIdentified Risks: {len(risk_results['identified_risks'])}")

for i, risk in enumerate(risk_results['identified_risks'], 1):
    print(f"\n{i}. {risk['category'].upper()} - {risk['severity']}")
    print(f"   {risk['description']}")

### Visualize Risk Distribution

In [ ]:
# Prepare risk data for visualization
risk_categories = [risk['category'] for risk in risk_results['identified_risks']]
risk_scores = [risk['likelihood'] for risk in risk_results['identified_risks']]

risk_df = pd.DataFrame({
    'Category': risk_categories,
    'Risk Score': risk_scores
})

fig = px.bar(
    risk_df,
    x='Category',
    y='Risk Score',
    title='Risk Distribution by Category',
    color='Risk Score',
    color_continuous_scale='Reds'
)

fig.show()

## 5. Complete Analysis Pipeline

Let's run a complete analysis for multiple stocks.

In [ ]:
def analyze_stock(symbol, days_back=7):
    """Complete analysis pipeline for a stock."""
    print(f"\n{'='*60}")
    print(f"Analyzing {symbol}")
    print(f"{'='*60}")
    
    # Research
    print("\n1. 🔍 Running research...")
    research = research_agent.execute({
        "symbol": symbol,
        "days_back": days_back
    })
    
    # Sentiment
    print("2. 💭 Analyzing sentiment...")
    sentiment = sentiment_agent.execute({
        "symbol": symbol,
        "text": research['findings']
    })
    
    # Risk
    print("3. ⚠️ Assessing risks...")
    risk = risk_agent.execute({
        "symbol": symbol,
        "news_data": research,
        "sentiment": sentiment
    })
    
    return {
        "symbol": symbol,
        "research": research,
        "sentiment": sentiment,
        "risk": risk
    }

# Analyze multiple stocks
symbols = ["AAPL", "GOOGL", "MSFT"]
results = {}

for symbol in symbols:
    try:
        results[symbol] = analyze_stock(symbol)
        print(f"\n✅ {symbol} analysis complete!")
    except Exception as e:
        print(f"\n❌ Error analyzing {symbol}: {str(e)}")

print("\n" + "="*60)
print("All analyses complete!")
print("="*60)

## 6. Comparative Analysis

Let's compare the results across different stocks.

In [ ]:
# Create comparison dataframe
comparison_data = []

for symbol, data in results.items():
    comparison_data.append({
        'Symbol': symbol,
        'Sentiment Score': data['sentiment']['sentiment_score'],
        'Risk Score': data['risk']['overall_risk_score'],
        'Sentiment': data['sentiment']['overall_sentiment'],
        'Risk Level': data['risk']['risk_level']
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n📊 Stock Comparison")
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Sentiment Score',
    x=comparison_df['Symbol'],
    y=comparison_df['Sentiment Score'],
    marker_color='lightblue'
))

fig.add_trace(go.Bar(
    name='Risk Score',
    x=comparison_df['Symbol'],
    y=comparison_df['Risk Score'],
    marker_color='lightcoral'
))

fig.update_layout(
    title='Sentiment vs Risk Comparison',
    barmode='group',
    yaxis_range=[0, 1]
)

fig.show()

## 7. Export Results

Let's export our analysis results.

In [ ]:
import json

# Export to JSON
output_file = f"../data/processed/analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(output_file, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"✅ Results exported to: {output_file}")

## Conclusion

This notebook demonstrated:
- ✅ Multi-agent architecture with LangChain
- ✅ Research, Sentiment, and Risk analysis
- ✅ Data visualization with Plotly
- ✅ Comparative stock analysis
- ✅ Results export

### Next Steps
- Try the Streamlit dashboard: `streamlit run streamlit_app/app.py`
- Explore the REST API: `uvicorn src.api.main:app --reload`
- Add custom agents and tools
- Build your own analysis workflows